***This page is dedicated for experimenting on the new algorithm to improve efficiency and scalability***

In [2]:
# Import Packages
import numpy as np
import cvxpy as cp
import pandas as pd
import matplotlib.pyplot as plt
import random
import math

# Import functions
from my_functions import *

In [3]:
def simple_sa(df_new, max_iter=5000, T0=1000, alpha=0.999):
    N = len(df_new)
    profit = df_new['Profit'].values
    size = df_new['size'].values
    cost = df_new['Buying Price'].values
    parkCapacity = 2 * np.sum(size) / 3
    dealBudget = np.sum(cost) / 4

    hybrid = df_new['fuel_type'].str.lower().isin(['hybrid','plug-in hybrid']).values
    imported = df_new['imported'] == 1

    # random feasible start
    current = np.random.randint(0, 2, N)
    while np.sum(size[current==1]) > parkCapacity or np.sum(cost[current==1]) > dealBudget:
        current[random.choice(np.where(current==1)[0])] = 0

    def evaluate(x):
        p = np.sum(profit[x==1])
        if np.any(hybrid & (x==1)):   p -= 300
        if np.any(imported & (x==1)): p -= 1000
        return p

    current_profit = evaluate(current)
    best, best_profit = current.copy(), current_profit
    temp = T0

    for _ in range(max_iter):
        idx = random.randint(0, N-1)
        new = current.copy()
        new[idx] = 1 - new[idx]

        # incremental update
        new_profit = evaluate(new)
        if np.sum(size[new==1]) <= parkCapacity and np.sum(cost[new==1]) <= dealBudget:
            delta = new_profit - current_profit
            if delta > 0 or random.random() < math.exp(delta / temp):
                current, current_profit = new, new_profit
                if new_profit > best_profit:
                    best, best_profit = new, new_profit
        temp = max(temp * alpha, 1e-3)

    print(f"Numberof cars chosen: {int(np.sum(best))}")
    print(f"Profit: ${best_profit:.2f}")
    print(f"Space taken: {np.sum(size[best==1]):.1f}/{parkCapacity:.1f} m^2")
    print(f"Cost: ${np.sum(cost[best==1]):.1f}/{dealBudget:.1f}")
    return best


df_new = pd.read_csv('data/complete_data.csv')

simple_sa(df_new)

Numberof cars chosen: 811
Profit: $6663626.40
Space taken: 7282.0/19294.7 m^2
Cost: $26706961.6/26708719.5


array([0, 0, 0, ..., 1, 0, 0], shape=(3223,), dtype=int32)